<a href="https://colab.research.google.com/github/NJerez-dev/Analisis-Datos-Transporte-UM/blob/main/01_bronze_ingesta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥉 Notebook 01 — Capa BRONZE


| Tabla creada | Fuente |
|---|---|
| `bronze_viajes_diarios` | Hoja `VIAJES DIARIOS` |
| `bronze_devoluciones` | Hoja `DEVOLUCIONES` |

## 0️⃣ Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE   = '/content/drive/MyDrive/SUPPLY_CHAIN_PROJECT'
EXCEL_FILE   = f'{DRIVE_BASE}/BD UM LA MEJOR EMPRESA DE TRANSPORTE.xlsx'
DB_PATH      = f'{DRIVE_BASE}/supply_chain.db'

print('✅ Drive montado')
print(f'📁 DB: {DB_PATH}')

Mounted at /content/drive
✅ Drive montado
📁 DB: /content/drive/MyDrive/SUPPLY_CHAIN_PROJECT/supply_chain.db


## 1️⃣ Importar librerías

In [2]:
import pandas as pd
import sqlite3
from datetime import datetime

print('✅ Librerías importadas')

✅ Librerías importadas


## 2️⃣ Leer Excel — Hoja VIAJES DIARIOS

In [3]:
EXCEL_FILE = '/content/drive/MyDrive/SUPPLY_CHAIN_PROJECT/BD UM LA MEJOR EMPRESA DE TRANSPORTE.xlsx' # Corrected path

df_viajes_raw = pd.read_excel(
    EXCEL_FILE,
    sheet_name='VIAJES DIARIOS',
    engine='openpyxl',
    dtype=str   # Bronze: todo como string, sin inferencia de tipos
)

# Agregar metadata de ingesta
df_viajes_raw['_ingesta_timestamp'] = datetime.now().isoformat()
df_viajes_raw['_fuente']            = 'VIAJES_DIARIOS'
df_viajes_raw['_archivo']           = 'BD UM LA MEJOR EMPRESA DE TRANSPORTE.xlsx'

print(f'📦 Filas leídas: {len(df_viajes_raw):,}')
print(f'📌 Columnas: {list(df_viajes_raw.columns)}')
df_viajes_raw.head(3)

📦 Filas leídas: 1,013
📌 Columnas: ['Suborden', 'Do', 'Estado', 'Patente', 'Empresa', 'Idruta', 'Posicionruta', 'Ct', 'Direccion', 'Localidad', 'Region', 'Volumen', 'Horainicio', 'Horafin', 'Fechainicioruta', 'Fechapactada', 'Fechaentregareal', 'Fechaestimada', 'Comentarionoentrega', 'Motivonoentrega', 'LPN', 'LPN_Container', 'Commerce', 'ParentOrder', 'Activa', '_ingesta_timestamp', '_fuente', '_archivo']


,Suborden,Do,Estado,Patente,Empresa,Idruta,Posicionruta,Ct,Direccion,Localidad,...,Comentarionoentrega,Motivonoentrega,LPN,LPN_Container,Commerce,ParentOrder,Activa,_ingesta_timestamp,_fuente,_archivo
0,100000000000000,100000000000000,Terminado,STKY45,LA MEJOR EMPRESA DE TRANSPORTES SPA,14130168,0,HUB XD,Avenida Manuel Rodriguez 694 203,SANTIAGO CENTRO,...,NaN,NaN,140111000013588362,910000000000907884,Falabella,3227404961,True,2026-04-09T01:59:07.195692,VIAJES_DIARIOS,BD UM LA MEJOR EMPRESA DE TRANSPORTE.xlsx
1,100000000000000,100000000000000,Terminado,STKY45,LA MEJOR EMPRESA DE TRANSPORTES SPA,14130168,1,HUB XD,Avenida Manuel Rodríguez 694 Depto 413,SANTIAGO CENTRO,...,NaN,NaN,140111000013605751,910000000000907884,Falabella,3227485971,True,2026-04-09T01:59:07.195692,VIAJES_DIARIOS,BD UM LA MEJOR EMPRESA DE TRANSPORTE.xlsx
2,100000000000000,100000000000000,Terminado,STKY45,LA MEJOR EMPRESA DE TRANSPORTES SPA,14130168,2,HUB XD,Santo domingo 1669 Departamento 801,SANTIAGO CENTRO,...,NaN,NaN,140111000013602528,910000000000907884,Falabella,3227471128,True,2026-04-09T01:59:07.195692,VIAJES_DIARIOS,BD UM LA MEJOR EMPRESA DE TRANSPORTE.xlsx


## 3️⃣ Leer Excel — Hoja DEVOLUCIONES

In [4]:
df_dev_raw = pd.read_excel(
    EXCEL_FILE,
    sheet_name='DEVOLUCIONES',
    engine='openpyxl',
    dtype=str
)

# Renombrar columna duplicada 'Estado' / 'Estado.1'
df_dev_raw = df_dev_raw.rename(columns={
    'N° Recepción'   : 'nro_recepcion',
    'Seller'         : 'seller',
    'Nº de Etiqueta' : 'nro_etiqueta',
    'Nº de OC'       : 'nro_oc',
    'Estado'         : 'estado_recepcion',
    'Observaciones'  : 'observaciones',
    'Patente'        : 'patente',
    'Fecha viaje'    : 'fecha_viaje',
    'Fecha devolución': 'fecha_devolucion',
    'Estado.1'       : 'estado_devolucion'
})

# Eliminar columnas vacías (None header)
df_dev_raw = df_dev_raw.loc[:, df_dev_raw.columns.notna()]

# Metadata de ingesta
df_dev_raw['_ingesta_timestamp'] = datetime.now().isoformat()
df_dev_raw['_fuente']            = 'DEVOLUCIONES'
df_dev_raw['_archivo']           = 'BD_UM_LA_MEJOR_EMPRESA_DE_TRANSPORTE.xlsx'

print(f'↩️  Filas leídas: {len(df_dev_raw):,}')
print(f'📌 Columnas: {list(df_dev_raw.columns)}')
df_dev_raw.head(3)

↩️  Filas leídas: 275
📌 Columnas: ['nro_recepcion', 'seller', 'nro_etiqueta', 'nro_oc', 'estado_recepcion', 'observaciones', 'patente', 'fecha_viaje', 'fecha_devolucion', 'estado_devolucion', '_ingesta_timestamp', '_fuente', '_archivo']


,nro_recepcion,seller,nro_etiqueta,nro_oc,estado_recepcion,observaciones,patente,fecha_viaje,fecha_devolucion,estado_devolucion,_ingesta_timestamp,_fuente,_archivo
0,CL1773091348393,Sin información,900600000002966557,-,Recepcionado,Tienda Errónea,STKY72,2026-03-02 00:00:00,2026-03-02 00:00:00,Devuelto,2026-04-09T01:59:07.624218,DEVOLUCIONES,BD_UM_LA_MEJOR_EMPRESA_DE_TRANSPORTE.xlsx
1,CL1773091348393,Sin información,140121000013560210,-,Recepcionado,Tienda Errónea,STKY72,2026-03-02 00:00:00,2026-03-02 00:00:00,Devuelto,2026-04-09T01:59:07.624218,DEVOLUCIONES,BD_UM_LA_MEJOR_EMPRESA_DE_TRANSPORTE.xlsx
2,CL1773091348393,Sin información,140121000013613608,-,Recepcionado,Tienda Errónea,STKY72,2026-03-02 00:00:00,2026-03-02 00:00:00,Devuelto,2026-04-09T01:59:07.624218,DEVOLUCIONES,BD_UM_LA_MEJOR_EMPRESA_DE_TRANSPORTE.xlsx


## 4️⃣ Escribir en SQLite — Capa Bronze

In [5]:
con = sqlite3.connect(DB_PATH)

# Cargar tablas Bronze (replace = recarga completa cada vez)
df_viajes_raw.to_sql('bronze_viajes_diarios', con, if_exists='replace', index=False)
df_dev_raw.to_sql('bronze_devoluciones',      con, if_exists='replace', index=False)

con.commit()
print('✅ Tablas Bronze escritas en SQLite')

✅ Tablas Bronze escritas en SQLite


## 5️⃣ Validación — Verificar tablas creadas

In [6]:
# Listar todas las tablas en la DB
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
print('📋 Tablas en supply_chain.db:')
print(tablas.to_string(index=False))

# Conteo de filas por tabla
print('\n📊 Conteo de filas:')
for tabla in tablas['name']:
    n = pd.read_sql(f'SELECT COUNT(*) as total FROM {tabla}', con).iloc[0,0]
    print(f'  {tabla}: {n:,} filas')

📋 Tablas en supply_chain.db:
                 name
  bronze_devoluciones
bronze_viajes_diarios
      gold_dim_tiempo
     gold_fact_viajes
gold_kpi_devoluciones
    gold_kpi_entregas
gold_kpi_por_commerce
 gold_kpi_por_patente
  silver_devoluciones
        silver_viajes

📊 Conteo de filas:
  bronze_devoluciones: 275 filas
  bronze_viajes_diarios: 1,013 filas
  gold_dim_tiempo: 1 filas
  gold_fact_viajes: 1,013 filas
  gold_kpi_devoluciones: 2 filas
  gold_kpi_entregas: 1 filas
  gold_kpi_por_commerce: 2 filas
  gold_kpi_por_patente: 7 filas
  silver_devoluciones: 275 filas
  silver_viajes: 1,013 filas


In [7]:
# Vista previa Bronze Viajes
print('--- bronze_viajes_diarios (3 filas) ---')
pd.read_sql('SELECT * FROM bronze_viajes_diarios LIMIT 3', con)

--- bronze_viajes_diarios (3 filas) ---


,Suborden,Do,Estado,Patente,Empresa,Idruta,Posicionruta,Ct,Direccion,Localidad,...,Comentarionoentrega,Motivonoentrega,LPN,LPN_Container,Commerce,ParentOrder,Activa,_ingesta_timestamp,_fuente,_archivo
0,100000000000000,100000000000000,Terminado,STKY45,LA MEJOR EMPRESA DE TRANSPORTES SPA,14130168,0,HUB XD,Avenida Manuel Rodriguez 694 203,SANTIAGO CENTRO,...,None,None,140111000013588362,910000000000907884,Falabella,3227404961,True,2026-04-09T01:59:07.195692,VIAJES_DIARIOS,BD UM LA MEJOR EMPRESA DE TRANSPORTE.xlsx
1,100000000000000,100000000000000,Terminado,STKY45,LA MEJOR EMPRESA DE TRANSPORTES SPA,14130168,1,HUB XD,Avenida Manuel Rodríguez 694 Depto 413,SANTIAGO CENTRO,...,None,None,140111000013605751,910000000000907884,Falabella,3227485971,True,2026-04-09T01:59:07.195692,VIAJES_DIARIOS,BD UM LA MEJOR EMPRESA DE TRANSPORTE.xlsx
2,100000000000000,100000000000000,Terminado,STKY45,LA MEJOR EMPRESA DE TRANSPORTES SPA,14130168,2,HUB XD,Santo domingo 1669 Departamento 801,SANTIAGO CENTRO,...,None,None,140111000013602528,910000000000907884,Falabella,3227471128,True,2026-04-09T01:59:07.195692,VIAJES_DIARIOS,BD UM LA MEJOR EMPRESA DE TRANSPORTE.xlsx


In [8]:
# Vista previa Bronze Devoluciones
print('--- bronze_devoluciones (3 filas) ---')
pd.read_sql('SELECT * FROM bronze_devoluciones LIMIT 3', con)

--- bronze_devoluciones (3 filas) ---


,nro_recepcion,seller,nro_etiqueta,nro_oc,estado_recepcion,observaciones,patente,fecha_viaje,fecha_devolucion,estado_devolucion,_ingesta_timestamp,_fuente,_archivo
0,CL1773091348393,Sin información,900600000002966557,-,Recepcionado,Tienda Errónea,STKY72,2026-03-02 00:00:00,2026-03-02 00:00:00,Devuelto,2026-04-09T01:59:07.624218,DEVOLUCIONES,BD_UM_LA_MEJOR_EMPRESA_DE_TRANSPORTE.xlsx
1,CL1773091348393,Sin información,140121000013560210,-,Recepcionado,Tienda Errónea,STKY72,2026-03-02 00:00:00,2026-03-02 00:00:00,Devuelto,2026-04-09T01:59:07.624218,DEVOLUCIONES,BD_UM_LA_MEJOR_EMPRESA_DE_TRANSPORTE.xlsx
2,CL1773091348393,Sin información,140121000013613608,-,Recepcionado,Tienda Errónea,STKY72,2026-03-02 00:00:00,2026-03-02 00:00:00,Devuelto,2026-04-09T01:59:07.624218,DEVOLUCIONES,BD_UM_LA_MEJOR_EMPRESA_DE_TRANSPORTE.xlsx


In [9]:
con.close()
print('🔒 Conexión cerrada')

🔒 Conexión cerrada
